In [13]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.feature_extraction import DictVectorizer
from sklearn.utils.validation import check_is_fitted
from sklearn.exceptions import NotFittedError
from sklearn.metrics import root_mean_squared_error
import mlflow

In [15]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("03-orchestration")

2025/06/08 18:40:10 INFO mlflow.tracking.fluent: Experiment with name '03-orchestration' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1749408010966, experiment_id='1', last_update_time=1749408010966, lifecycle_stage='active', name='03-orchestration', tags={}>

In [7]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)
    print("number of rows before wrangling", df.shape[0])

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.dt.total_seconds() / 60

    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    print("number of rows after wrangling", df.shape[0])


    return df

In [8]:
filename = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-03.parquet"

In [9]:
df = read_dataframe(filename)

number of rows before wrangling 3403766
number of rows after wrangling 3316216


In [17]:
def features_selection_and_encoding(df, encoder):
    
    categorical = ['PULocationID', 'DOLocationID']
    #numerical = ["trip_distance"]
    
    features = df[categorical]
    
    df_dicts = features.to_dict(orient="records")
    
    try:
        check_is_fitted(encoder)
        X = encoder.transform(df_dicts)
    except NotFittedError:
        X = encoder.fit_transform(df_dicts)

    target = "duration"
    
    y = df[target]

    return X, y, dv

In [18]:
dv = DictVectorizer()

In [19]:
X, y, dv = features_selection_and_encoding(df, dv)

In [16]:
def train_model(X, y, encoder):
    with mlflow.start_run():
        lr = LinearRegression()
        lr.fit(X, y)
        print("model intercept", lr.intercept_)
        mlflow.log_metric("intercept_i", lr.intercept_)
        mlflow.sklearn.log_model(encoder, "Dict Vectorizer")
        mlflow.sklearn.log_model(lr, "linear regression")
    #root_mean_squared_error(y_train, y_pred)

In [20]:
train_model(X, y, dv)

2025/06/08 18:59:07 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


model intercept 24.774419321244913


2025/06/08 18:59:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/06/08 18:59:14 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run merciful-trout-669 at: http://localhost:5000/#/experiments/1/runs/0821ba3feb9c49f38564cf6519fd77fa
🧪 View experiment at: http://localhost:5000/#/experiments/1
